# SSL-2: Barlow Twins
**PM25Vision | Tasks 4–6** | Backbone: EfficientNet-B0 | Batch: 32 | Epochs: 30

In [1]:
# Cell 0 — GPU Reset
import gc, torch, os, warnings
warnings.filterwarnings("ignore")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect(); torch.cuda.synchronize()
    free  = torch.cuda.mem_get_info()[0]/1e9
    total = torch.cuda.mem_get_info()[1]/1e9
    print(f"GPU: {total:.1f} GB total | {total-free:.2f} GB used | {free:.2f} GB free")
else:
    print("No GPU — running on CPU.")


GPU: 15.6 GB total | 0.11 GB used | 15.53 GB free


In [2]:
# Cell 1 — Install
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q",
    "timm","thop","scikit-learn","umap-learn",
    "matplotlib","seaborn","scipy"], check=True)
print("Installs done.")


Installs done.


In [3]:
# Cell 2 — Imports
import os, gc, time, random, warnings, copy
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
from thop import profile as thop_profile

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve, classification_report)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats
import umap as umap_lib

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")


2026-04-18 17:31:25.516296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776533485.898391      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776533486.019894      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776533486.996784      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776533486.996826      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776533486.996829      22 computation_placer.cc:177] computation placer alr

Device: cuda | PyTorch: 2.10.0+cu128


In [4]:
# Cell 3 — Config
SSL_NAME        = "BarlowTwins"
BACKBONE        = "efficientnet_b0"
BATCH_SIZE      = 32
PRETRAIN_EPOCHS = 30
IMG_SIZE        = 224
LAMBD           = 0.0051   # off-diagonal weight
PROJ_DIM        = 2048
print(f"SSL: {SSL_NAME} | backbone: {BACKBONE} | batch: {BATCH_SIZE} | lambda: {LAMBD}")


SSL: BarlowTwins | backbone: efficientnet_b0 | batch: 32 | lambda: 0.0051


In [5]:
# Cell 4 — Data loading
TRAIN_CSV = "/kaggle/input/datasets/deadcardassian/pm25vision/train/metadata.csv"
TEST_CSV  = "/kaggle/input/datasets/deadcardassian/pm25vision/test/metadata.csv"
TRAIN_IMG = "/kaggle/input/datasets/deadcardassian/pm25vision/train/images"
TEST_IMG  = "/kaggle/input/datasets/deadcardassian/pm25vision/test/images"

train_df = pd.read_csv(TRAIN_CSV); test_df = pd.read_csv(TEST_CSV)

def detect_cols(df):
    fc = next((c for c in df.columns if df[c].astype(str)
               .str.contains(r"\.(jpg|jpeg|png)", case=False, regex=True).any()), df.columns[0])
    lc = next((c for c in df.columns if c.lower() in
               ("bin","class","label","aqi","category","pm25_bin","aqi_bin","level")), df.columns[-1])
    return fc, lc

tr_f, tr_l = detect_cols(train_df); te_f, te_l = detect_cols(test_df)
all_labels  = sorted(pd.concat([train_df[tr_l], test_df[te_l]]).unique())
label_map   = {orig:i for i,orig in enumerate(all_labels)}
train_df["_label"] = train_df[tr_l].map(label_map)
test_df["_label"]  = test_df[te_l].map(label_map)
NUM_CLASSES = len(all_labels); CLASS_NAMES = [str(k) for k in all_labels]
full_df = pd.concat([train_df, test_df], ignore_index=True)
n_orig  = len(train_df)
full_df["_img_dir"] = [TRAIN_IMG if i < n_orig else TEST_IMG for i in range(len(full_df))]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class PM25DS(Dataset):
    def __init__(self, df, fc, tf):
        self.df=df.reset_index(drop=True); self.fc=fc; self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        img=Image.open(os.path.join(row["_img_dir"],os.path.basename(str(row[self.fc])))).convert("RGB")
        return self.tf(img), int(row["_label"])

print(f"Pool: {len(full_df)} images | {NUM_CLASSES} classes: {CLASS_NAMES}")


Pool: 11219 images | 6 classes: ['0–50', '101–150', '151–200', '201–300', '301–600', '51–100']


In [6]:
# Cell 5 — Augmentation recipe (Barlow Twins symmetric, gentle for PM2.5 haze preservation)
# Both views use same transform — gentle ColorJitter preserves haze color signal
# Scale (0.4, 1.0) keeps global context for AQI estimation
bt_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)], p=0.6),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(23, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class BarlowDS(Dataset):
    def __init__(self, df, fc, tf):
        self.df=df.reset_index(drop=True); self.fc=fc; self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        img=Image.open(os.path.join(row["_img_dir"],os.path.basename(str(row[self.fc])))).convert("RGB")
        return self.tf(img), self.tf(img), int(row["_label"])

pretrain_ds  = BarlowDS(full_df, tr_f, bt_tf)
pretrain_ldr = DataLoader(pretrain_ds, BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
print(f"Pretrain loader: {len(pretrain_ldr)} batches/epoch")


Pretrain loader: 350 batches/epoch


In [7]:
# Cell 6 — Barlow Twins model
class BarlowTwins(nn.Module):
    def __init__(self, backbone=BACKBONE, proj_dim=PROJ_DIM, lambd=LAMBD):
        super().__init__(); self.lambd = lambd
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        fd = self.backbone.num_features
        # 3-layer projector with BN
        self.projector = nn.Sequential(
            nn.Linear(fd, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(),
            nn.Linear(proj_dim, proj_dim), nn.BatchNorm1d(proj_dim), nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.bn = nn.BatchNorm1d(proj_dim, affine=False)

    def off_diagonal(self, x):
        n, m = x.shape
        return x.flatten()[:-1].view(n-1, n+1)[:, 1:].flatten()

    def forward(self, y1, y2):
        z1 = self.projector(self.backbone(y1))
        z2 = self.projector(self.backbone(y2))
        c = self.bn(z1).T @ self.bn(z2)
        c.div_(z1.shape[0])
        on_diag  = torch.diagonal(c).add_(-1).pow_(2).sum()
        off_diag = self.off_diagonal(c).pow_(2).sum()
        return on_diag + self.lambd * off_diag

model = BarlowTwins().to(DEVICE)
dummy = torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE)
with torch.no_grad():
    macs,_ = thop_profile(model.backbone, inputs=(dummy,), verbose=False)
gflops = macs/1e9
total_params = sum(p.numel() for p in model.parameters())/1e6
print(f"Params: {total_params:.1f}M | GFLOPs: {gflops:.3f}")
torch.cuda.empty_cache()


Params: 15.0M | GFLOPs: 0.385


In [8]:
# Cell 7 — Pretraining (Task 5.1)
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRETRAIN_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

losses = []; pretrain_time_start = time.time()
for ep in range(1, PRETRAIN_EPOCHS+1):
    model.train(); ep_l=0; nb=0
    for x1,x2,_ in pretrain_ldr:
        x1,x2=x1.to(DEVICE),x2.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss=model(x1,x2)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        ep_l+=loss.item(); nb+=1
    sch.step(); avg=ep_l/nb; losses.append(avg)
    if ep%5==0 or ep==1:
        mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
        print(f"Ep {ep:03d}/{PRETRAIN_EPOCHS} loss={avg:.4f} gpu={mem:.2f}GB")

pretrain_time = time.time()-pretrain_time_start
torch.cuda.empty_cache()
torch.save(model.state_dict(), "barlow_twins_encoder.pt")
print(f"Saved: barlow_twins_encoder.pt | {pretrain_time:.0f}s ({pretrain_time/60:.1f} min)")

plt.figure(figsize=(8,4))
plt.plot(losses, lw=2); plt.xlabel("Epoch"); plt.ylabel("Barlow Twins Loss")
plt.title("Barlow Twins Pretraining Loss"); plt.grid(True,alpha=0.5)
plt.tight_layout(); plt.savefig("pretrain_loss_BarlowTwins.png",dpi=150); plt.show()
print("Loss curve saved.")

encoder = model.backbone
for p in encoder.parameters(): p.requires_grad_(False)
encoder.eval(); torch.cuda.empty_cache()
print("Encoder frozen.")


Ep 001/30 loss=2757.8782 gpu=0.30GB
Ep 005/30 loss=2341.3145 gpu=0.30GB
Ep 010/30 loss=2206.8820 gpu=0.30GB
Ep 015/30 loss=2161.2389 gpu=0.30GB
Ep 020/30 loss=2117.7916 gpu=0.30GB
Ep 025/30 loss=2137.3504 gpu=0.30GB
Ep 030/30 loss=2118.4636 gpu=0.30GB
Saved: barlow_twins_encoder.pt | 11545s (192.4 min)
Loss curve saved.
Encoder frozen.


In [9]:
# Cell 8 — Feature extraction
def extract_features(enc, loader):
    enc.eval(); feats, lbls = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                feats.append(enc(x.to(DEVICE)).cpu().float().numpy())
            lbls.extend(y.numpy())
    return np.concatenate(feats), np.array(lbls)

tr_idx, te_idx = train_test_split(
    list(range(len(full_df))), test_size=0.20,
    stratify=full_df["_label"].tolist(), random_state=SEED)

train_ldr_ev = DataLoader(PM25DS(full_df.iloc[tr_idx], tr_f, eval_tf), 64, num_workers=2, pin_memory=True)
test_ldr_ev  = DataLoader(PM25DS(full_df.iloc[te_idx], tr_f, eval_tf), 64, num_workers=2, pin_memory=True)

print("Extracting frozen features...")
t0 = time.time()
X_tr, y_tr = extract_features(encoder, train_ldr_ev)
X_te, y_te = extract_features(encoder, test_ldr_ev)
feat_time = time.time()-t0
print(f"Train {X_tr.shape} | Test {X_te.shape} | {feat_time:.1f}s")

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)


Extracting frozen features...
Train (8975, 1280) | Test (2244, 1280) | 102.0s


In [10]:
# Cell 9 — Shallow heads (Task 5.2)
def eval_head(name, clf):
    t0=time.time(); clf.fit(X_tr_s, y_tr); tr_t=time.time()-t0
    t0=time.time(); preds=clf.predict(X_te_s); te_t=time.time()-t0
    acc=accuracy_score(y_te,preds)
    f1 =f1_score(y_te,preds,average="macro",zero_division=0)
    try:
        prob=clf.predict_proba(X_te_s)
        auc=roc_auc_score(y_te,prob,multi_class="ovr",average="macro")
    except: auc=float("nan")
    print(f"  {name:22s} acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}  fit={tr_t:.1f}s")
    return {"head":name,"acc":acc,"f1":f1,"auc":auc,"train_t":round(tr_t,2),"test_t":round(te_t,3)}

print("── Shallow heads on frozen features ──")
heads = []
heads.append(eval_head("Linear Probe",    LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)))
heads.append(eval_head("MLP",             MLPClassifier(hidden_layer_sizes=(256,128), max_iter=200, random_state=SEED)))
heads.append(eval_head("SVM (RBF)",       SVC(kernel="rbf", C=10, probability=True, random_state=SEED)))
heads.append(eval_head("Decision Tree",   DecisionTreeClassifier(max_depth=12, random_state=SEED)))
heads.append(eval_head("Random Forest",   RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)))
df_heads = pd.DataFrame(heads)
print(); print(df_heads[["head","acc","f1","auc"]].to_string(index=False))


── Shallow heads on frozen features ──
  Linear Probe           acc=0.2986  f1=0.2409  auc=0.6348  fit=93.3s
  MLP                    acc=0.2692  f1=0.2301  auc=0.6000  fit=47.9s
  SVM (RBF)              acc=0.2785  f1=0.2067  auc=0.6042  fit=358.0s
  Decision Tree          acc=0.2687  f1=0.2424  auc=0.5773  fit=16.2s
  Random Forest          acc=0.2901  f1=0.2791  auc=0.6308  fit=18.2s

         head      acc       f1      auc
 Linear Probe 0.298574 0.240949 0.634793
          MLP 0.269162 0.230122 0.600005
    SVM (RBF) 0.278520 0.206669 0.604218
Decision Tree 0.268717 0.242371 0.577292
Random Forest 0.290107 0.279149 0.630811


In [11]:
# Cell 10 — Full fine-tuning vs linear probe (Task 5.2 — was MISSING)
print("── Full fine-tuning ──")
FINETUNE_EPOCHS = 20; LR_FT = 1e-4

class FTModel(nn.Module):
    def __init__(self, enc, feat_dim, n_cls):
        super().__init__(); self.enc=enc; self.head=nn.Linear(feat_dim, n_cls)
    def forward(self, x): return self.head(self.enc(x))

for p in encoder.parameters(): p.requires_grad_(True)
ft_model  = FTModel(encoder, X_tr.shape[1], NUM_CLASSES).to(DEVICE)
ft_opt    = optim.AdamW(ft_model.parameters(), lr=LR_FT, weight_decay=1e-4)
ft_sch    = optim.lr_scheduler.CosineAnnealingLR(ft_opt, T_max=FINETUNE_EPOCHS)
ft_crit   = nn.CrossEntropyLoss()
ft_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
ft_aug    = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.3,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
ft_train_ldr = DataLoader(PM25DS(full_df.iloc[tr_idx], tr_f, ft_aug),
                          batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

ft_hist = {"loss":[],"acc":[]}
ft_wall_start = time.time()
for ep in range(1, FINETUNE_EPOCHS+1):
    ft_model.train(); ep_l=0; ep_c=0; ep_n=0
    for x,y in ft_train_ldr:
        x,y=x.to(DEVICE),y.to(DEVICE); ft_opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out=ft_model(x); loss=ft_crit(out,y)
        ft_scaler.scale(loss).backward(); ft_scaler.step(ft_opt); ft_scaler.update()
        ep_l+=loss.item(); ep_c+=(out.argmax(1)==y).sum().item(); ep_n+=y.size(0)
    ft_sch.step()
    ft_hist["loss"].append(ep_l/len(ft_train_ldr)); ft_hist["acc"].append(ep_c/ep_n)
    if ep%5==0 or ep==1:
        print(f"  FT ep {ep:02d}/{FINETUNE_EPOCHS}  loss={ep_l/len(ft_train_ldr):.4f}  acc={ep_c/ep_n:.4f}")
ft_wall_time = time.time()-ft_wall_start

ft_model.eval(); ft_preds=[]; ft_probs=[]
with torch.no_grad():
    for x,y in test_ldr_ev:
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits=ft_model(x.to(DEVICE))
        ft_preds.extend(logits.argmax(1).cpu().numpy())
        ft_probs.extend(torch.softmax(logits,1).cpu().float().numpy())
ft_preds=np.array(ft_preds); ft_probs=np.array(ft_probs)
ft_acc=accuracy_score(y_te,ft_preds)
ft_f1=f1_score(y_te,ft_preds,average="macro",zero_division=0)
try:    ft_auc=roc_auc_score(y_te,ft_probs,multi_class="ovr",average="macro")
except: ft_auc=float("nan")
lp_acc=heads[0]["acc"]
print(f"  Linear Probe acc : {lp_acc:.4f}")
print(f"  Fine-Tuned  acc  : {ft_acc:.4f}  (Δ = {ft_acc-lp_acc:+.4f})")
print(f"  Fine-Tuned  F1   : {ft_f1:.4f} | AUC: {ft_auc:.4f}")
print(f"  Fine-tune wall   : {ft_wall_time:.0f}s")

fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
a1.plot(ft_hist["loss"],lw=2); a1.set_title("Fine-tune Loss"); a1.set_xlabel("Epoch"); a1.grid(True,alpha=0.5)
a2.plot(ft_hist["acc"],lw=2,color="teal"); a2.set_title("Fine-tune Train Acc"); a2.set_xlabel("Epoch"); a2.grid(True,alpha=0.5)
plt.suptitle(f"{SSL_NAME} Fine-tuning Curves"); plt.tight_layout()
plt.savefig(f"finetune_curves_{SSL_NAME}.png",dpi=150); plt.show()

for p in encoder.parameters(): p.requires_grad_(False)
encoder.eval(); torch.cuda.empty_cache()


── Full fine-tuning ──
  FT ep 01/20  loss=1.6846  acc=0.2492
  FT ep 05/20  loss=1.5826  acc=0.3072
  FT ep 10/20  loss=1.5041  acc=0.3513
  FT ep 15/20  loss=1.4476  acc=0.3736
  FT ep 20/20  loss=1.4285  acc=0.3926
  Linear Probe acc : 0.2986
  Fine-Tuned  acc  : 0.4332  (Δ = +0.1346)
  Fine-Tuned  F1   : 0.4500 | AUC: nan
  Fine-tune wall   : 1879s


In [12]:
# Cell 11 — k-NN & label efficiency (Task 5.4)
print("── k-NN in embedding space ──")
knn_res = []
for k in [1, 5, 20]:
    knn=KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(X_tr_s, y_tr)
    acc_k=accuracy_score(y_te, knn.predict(X_te_s))
    print(f"  k={k:2d}: {acc_k:.4f}"); knn_res.append({"k":k,"acc":round(acc_k,4)})
df_knn = pd.DataFrame(knn_res)

print("── Label-efficiency curve ──")
le_res = []
for frac in [0.01, 0.05, 0.10, 0.25, 0.50]:
    n = max(NUM_CLASSES, int(frac*len(X_tr)))
    idx2, _ = train_test_split(range(len(X_tr)), train_size=n, stratify=y_tr, random_state=SEED)
    lr2=LogisticRegression(max_iter=500, C=1.0, random_state=SEED)
    lr2.fit(X_tr_s[idx2], y_tr[idx2])
    acc2=accuracy_score(y_te, lr2.predict(X_te_s))
    print(f"  {int(frac*100):3d}% ({n:4d} samples): {acc2:.4f}")
    le_res.append({"pct":int(frac*100),"n":n,"acc":round(acc2,4)})
df_le = pd.DataFrame(le_res)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
ax1.bar([r["head"] for r in heads],[r["acc"] for r in heads], edgecolor="k",
        color=plt.cm.tab10(np.linspace(0,0.5,len(heads))))
ax1.set_xticklabels([r["head"] for r in heads],rotation=25,ha="right",fontsize=9)
ax1.set_ylabel("Accuracy"); ax1.set_title(f"{SSL_NAME} — Frozen Feature Heads"); ax1.grid(axis="y",alpha=0.4)
ax2.plot(df_le["pct"], df_le["acc"], "o-", lw=2, ms=7)
ax2.set_xlabel("Labeled data (%)"); ax2.set_ylabel("Linear Probe Accuracy")
ax2.set_title(f"{SSL_NAME} — Label-Efficiency"); ax2.grid(True,alpha=0.4)
plt.tight_layout(); plt.savefig(f"heads_le_{SSL_NAME}.png",dpi=150); plt.show()


── k-NN in embedding space ──
  k= 1: 0.2531
  k= 5: 0.2647
  k=20: 0.2620
── Label-efficiency curve ──
    1% (  89 samples): 0.2094
    5% ( 448 samples): 0.2678
   10% ( 897 samples): 0.2839
   25% (2243 samples): 0.2857
   50% (4487 samples): 0.2812


In [13]:
# Cell 12 — Full metrics: per-class acc, confusion matrix, ROC (Task 5.4 — was MISSING)
best_clf = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
best_clf.fit(X_tr_s, y_tr)
preds = best_clf.predict(X_te_s)
prob  = best_clf.predict_proba(X_te_s)

acc      = accuracy_score(y_te, preds)
prec_mac = precision_score(y_te, preds, average="macro", zero_division=0)
rec_mac  = recall_score(y_te, preds,    average="macro", zero_division=0)
f1_mac   = f1_score(y_te, preds,        average="macro", zero_division=0)
try:    roc_auc = roc_auc_score(y_te, prob, multi_class="ovr", average="macro")
except: roc_auc = float("nan")
cm = confusion_matrix(y_te, preds)
pca_acc = cm.diagonal() / cm.sum(axis=1)

print("="*58)
for lbl, val in [("Accuracy",acc),("Macro Precision",prec_mac),
                 ("Macro Recall",rec_mac),("Macro F1",f1_mac),("ROC-AUC",roc_auc)]:
    print(f"  {lbl:20s}: {val:.4f}")
print(f"  {'Pretrain time':20s}: {pretrain_time:.0f}s")
print(f"  {'Fine-tune time':20s}: {ft_wall_time:.0f}s")
print(f"  {'GFLOPs':20s}: {gflops:.3f}")
print("="*58)
print("Per-class accuracy:")
for n,pa in zip(CLASS_NAMES,pca_acc): print(f"  Class {n}: {pa:.4f}")
print(); print(classification_report(y_te, preds, target_names=CLASS_NAMES))

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1, linewidths=0.5)
ax1.set_title(f"{SSL_NAME} — Confusion Matrix (Linear Probe)")
ax1.set_xlabel("Predicted"); ax1.set_ylabel("True")
lbls_bin = label_binarize(y_te, classes=list(range(NUM_CLASSES)))
for i in range(NUM_CLASSES):
    try:
        fpr,tpr,_=roc_curve(lbls_bin[:,i],prob[:,i])
        auc_i=roc_auc_score(lbls_bin[:,i],prob[:,i])
        ax2.plot(fpr,tpr,lw=2,label=f"Class {CLASS_NAMES[i]} ({auc_i:.3f})")
    except: pass
ax2.plot([0,1],[0,1],"k--",lw=1); ax2.legend(fontsize=8); ax2.grid(True,alpha=0.4)
ax2.set_title(f"{SSL_NAME} — ROC Curves"); ax2.set_xlabel("FPR"); ax2.set_ylabel("TPR")
plt.tight_layout(); plt.savefig(f"metrics_{SSL_NAME}.png",dpi=150); plt.show()


  Accuracy            : 0.2986
  Macro Precision     : 0.2653
  Macro Recall        : 0.2473
  Macro F1            : 0.2409
  ROC-AUC             : 0.6348
  Pretrain time       : 11545s
  Fine-tune time      : 1879s
  GFLOPs              : 0.385
Per-class accuracy:
  Class 0–50: 0.3071
  Class 101–150: 0.2988
  Class 151–200: 0.5353
  Class 201–300: 0.1727
  Class 301–600: 0.0000
  Class 51–100: 0.1701

              precision    recall  f1-score   support

        0–50       0.31      0.31      0.31       482
     101–150       0.28      0.30      0.29       482
     151–200       0.33      0.54      0.41       482
     201–300       0.46      0.17      0.25       220
     301–600       0.00      0.00      0.00        96
      51–100       0.21      0.17      0.19       482

    accuracy                           0.30      2244
   macro avg       0.27      0.25      0.24      2244
weighted avg       0.29      0.30      0.28      2244



In [14]:
# Cell 13 — Embedding analysis: PCA, t-SNE, UMAP, Silhouette (Task 5.3)
N = min(2000, len(X_te))
idx_p = np.random.choice(len(X_te), N, replace=False)
Xp = X_te_s[idx_p]; yp = y_te[idx_p]
pal = plt.cm.tab10(np.linspace(0,0.6,NUM_CLASSES))

print("Running PCA..."); Xpca  = PCA(2, random_state=SEED).fit_transform(Xp)
print("Running t-SNE..."); Xtsne = TSNE(2, random_state=SEED, perplexity=40, n_iter=1000).fit_transform(Xp)
print("Running UMAP..."); Xumap = umap_lib.UMAP(n_components=2, random_state=SEED, n_neighbors=30).fit_transform(Xp)
sil = silhouette_score(Xp, yp)
print(f"Silhouette score: {sil:.4f}")

fig,axes = plt.subplots(1,3,figsize=(18,5))
for ax, X2, title in zip(axes,[Xpca,Xtsne,Xumap],["PCA","t-SNE","UMAP"]):
    for c in range(NUM_CLASSES):
        m = yp==c
        ax.scatter(X2[m,0],X2[m,1],s=8,alpha=0.6,color=pal[c],label=CLASS_NAMES[c])
    ax.set_title(f"{SSL_NAME} — {title}"); ax.legend(fontsize=7,markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"Silhouette = {sil:.4f}  |  {SSL_NAME} frozen encoder", fontsize=12)
plt.tight_layout(); plt.savefig(f"embeddings_{SSL_NAME}.png",dpi=150); plt.show()


Running PCA...
Running t-SNE...
Running UMAP...
Silhouette score: -0.0459


In [15]:
# Cell 16 — Save complete summary CSV
summary = {
    "SSL":SSL_NAME, "Backbone":BACKBONE,
    "Accuracy":round(acc,4), "Macro Precision":round(prec_mac,4),
    "Macro Recall":round(rec_mac,4), "Macro F1":round(f1_mac,4),
    "Macro ROC-AUC":round(float(roc_auc),4), "Silhouette":round(sil,4),
    "FT Accuracy":round(ft_acc,4), "FT F1":round(ft_f1,4),
    "LP vs FT delta":round(ft_acc-acc,4),
    "kNN-1": df_knn[df_knn.k==1]["acc"].values[0],
    "kNN-5": df_knn[df_knn.k==5]["acc"].values[0],
    "kNN-20":df_knn[df_knn.k==20]["acc"].values[0],
    "Pretrain Time (s)":round(pretrain_time,1),
    "FineTune Time (s)":round(ft_wall_time,1),
    "GFLOPs":round(gflops,3),
}
pd.DataFrame([summary]).to_csv(f"ssl_summary_{SSL_NAME}.csv", index=False)
df_heads.to_csv(f"heads_{SSL_NAME}.csv", index=False)
df_le.to_csv(f"label_eff_{SSL_NAME}.csv", index=False)
print("All files saved:")
for k,v in summary.items(): print(f"  {k:25s}: {v}")


All files saved:
  SSL                      : BarlowTwins
  Backbone                 : efficientnet_b0
  Accuracy                 : 0.2986
  Macro Precision          : 0.2653
  Macro Recall             : 0.2473
  Macro F1                 : 0.2409
  Macro ROC-AUC            : 0.6348
  Silhouette               : -0.045899998396635056
  FT Accuracy              : 0.4332
  FT F1                    : 0.45
  LP vs FT delta           : 0.1346
  kNN-1                    : 0.2531
  kNN-5                    : 0.2647
  kNN-20                   : 0.262
  Pretrain Time (s)        : 11544.7
  FineTune Time (s)        : 1879.3
  GFLOPs                   : 0.385
